# 11 추가 실습 — LangGraph로 만드는 질문 라우팅 RAG

앞에서는 LangChain 에이전트가 **도구를 스스로 골랐습니다.** 이번에는 개발자가 **가능한 경로를 그래프로 먼저 정하고**, 모델은 질문을 분류만 합니다. 실제로 어느 경로를 탈지는 분류 결과로 정해집니다.

```
START → 질문 분류(classify)
          ├─ route = "general"  → 직접 답변(direct)  → END
          └─ route = "document" → 문서 검색(retrieve) → 근거 답변(grounded) → END
```

**실행 전에 예상해 보세요.** 같은 챗봇에 아래 세 질문을 보내면 각각 어느 경로로 갈까요? 마지막 5절에서 실제로 실행된 노드와 비교합니다.

| 질문 | 내 예상 경로 |
|---|---|
| 안녕하세요? | |
| 감광액이 덮이지 않은 부분을 없애는 단계는? | |
| 우리 회사 A라인의 어제 식각 불량 원인은 뭐야? | |

- 문서: 삼일PwC 『반도체 산업 Business Guidebook』(2025.12) **책자 6쪽 = PDF 8번째 페이지** ‘2. 반도체 Value Chain Overview (8대 공정)’ · [발행처 원문(8번째 페이지)](https://www.pwc.com/kr/ko/insights/industry-focus/samilpwc_semicon-guidebook.pdf#page=8)
- 원본 PDF는 함께 배포하지 않습니다. 노트북 옆 `data/samilpwc_semicon-guidebook.pdf`가 없으면 발행처 주소에서 내려받습니다. 내려받을 수 없는 환경이면 위 링크에서 직접 받아 그 위치에 두세요.
- 앞 챕터를 실행하지 않아도 됩니다. 새 커널에서 첫 셀부터 차례로 실행합니다.
- 이 노트북은 질문 **한 건**을 처리하는 그래프만 다룹니다. 대화 메모리와 추가 에이전트는 넣지 않습니다.
- 채팅 모델 gpt-6-luna와 임베딩 모델 text-embedding-3-small을 씁니다(키 하나, 주소 두 개).

In [ ]:
# 현재 Python 3.12 커널에 실습 패키지를 설치합니다(처음 한 번). 09 노트북과 같은 조합에 langgraph를 더했습니다.
# 설치 후 커널 재시작 안내가 나오면 재시작하고 다음 셀부터 실행하세요.
%pip install langchain-core==1.8.4 langchain-openai==1.8.5 langchain-chroma==1.2.0 chromadb==1.5.9 openai==3.59.0 pypdf==7.1.1 langgraph==1.2.24

In [ ]:
# ▶ 설정 셀 — 따옴표 안의 값만 바꾸세요. 모델명은 확정값이라 바꾸지 않습니다.
# 키가 든 이 셀은 화면 공유·캡처하거나 AI 도우미에 붙여넣지 않습니다. 파일을 공유할 때는 자리표시 문자열로 되돌립니다.
API_KEY = "여기에_수업용_API_키_입력"
BASE_URL = "여기에_채팅용_BASE_URL_입력"                # 안내받은 채팅 모델 주소(https://...)
MODEL_NAME = "gpt-6-luna"                              # 채팅·도구 호출·에이전트 모델(확정값)
EMBEDDING_BASE_URL = "여기에_임베딩용_BASE_URL_입력"    # 안내받은 임베딩 모델 주소(https://..., 채팅 주소와 다름)
EMBEDDING_MODEL_NAME = "text-embedding-3-small"        # 문서·질문 임베딩 모델(확정값)

In [ ]:
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
NEEDED = ["API_KEY", "BASE_URL", "EMBEDDING_BASE_URL"]
missing = [name for name in NEEDED if globals()[name].startswith("여기에") or not globals()[name].strip()]
if missing:   # 임베딩 주소가 없을 때 채팅 주소로 대신하지 않는다
    raise ValueError(", ".join(missing) + "에 안내받은 값을 설정 셀의 따옴표 안에 입력한 뒤 다시 실행하세요. (모델을 호출하기 전에 멈췄습니다)")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL, timeout=60, max_retries=1)            # 채팅 주소
json_llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL, timeout=60, max_retries=1,
                      model_kwargs={"response_format": {"type": "json_object"}})   # JSON 객체 하나만 답하는 모델
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME, api_key=API_KEY, base_url=EMBEDDING_BASE_URL,   # 임베딩 주소
                              check_embedding_ctx_length=False)
print("채팅 모델:", MODEL_NAME, "· 임베딩 모델:", EMBEDDING_MODEL_NAME)

## 1. 문서 준비 — 09와 같은 코드를 한 셀로

PDF 8번째 페이지(책자 6쪽) 본문 → 200자·겹침 60자 조각 → Chroma 색인. 조각마다 `chunk_id`·문서명·쪽수가 metadata로 붙습니다. 이미 색인이 있으면 다시 저장하지 않습니다. 이 노트북의 중심은 2절부터입니다.

In [ ]:
import os, urllib.request
from pathlib import Path

PDF_URL = "https://www.pwc.com/kr/ko/insights/industry-focus/samilpwc_semicon-guidebook.pdf"
PDF_FILE = Path("data/samilpwc_semicon-guidebook.pdf")       # 원본은 함께 배포하지 않음 → 없으면 내려받음
SOURCE = "삼일PwC 『반도체 산업 Business Guidebook』(2025.12)"
PAGE_INDEX = 7                       # 파이썬은 0부터 셈 → PDF 뷰어의 8번째 페이지
PDF_PAGE, BOOK_PAGE = PAGE_INDEX + 1, 6   # 책자에 인쇄된 쪽 번호는 6
VIEW_URL = f"{PDF_URL}#page={PDF_PAGE}"   # #page=8은 뷰어 이동용. 코드에서는 PAGE_INDEX로 페이지를 고른다

def ensure_pdf():
    if not PDF_FILE.is_file():
        PDF_FILE.parent.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(PDF_URL, timeout=60) as r:     # 기본 요청으로 받는다(브라우저 흉내 헤더는 거절됨)
                PDF_FILE.write_bytes(r.read())
            print("발행처에서 내려받음:", PDF_URL)
        except Exception as e:
            raise RuntimeError(f"PDF를 받지 못했습니다({type(e).__name__}). 브라우저로 {PDF_URL} 을 내려받아 "
                               f"이 노트북 옆 {PDF_FILE} 에 두고 다시 실행하세요.")
    return PDF_FILE

from pypdf import PdfReader

def page_runs(page):
    """페이지의 글자 묶음마다 (글자 크기, 세로 위치, 글자)를 모은다."""
    runs = []
    page.extract_text(visitor_text=lambda t, cm, tm, fd, fs: runs.append((round(fs * (tm[0] or 1), 1), tm[5], t)))
    return runs

def read_body(page):
    """제목(20pt)과 본문(10pt, 공정 그림 아래쪽)만 남기고 그림 속 짧은 이름표는 뺀다."""
    runs = page_runs(page)
    title = "".join(t for size, y, t in runs if size >= 19).replace("\n", "").strip()
    body = "".join(t for size, y, t in runs if 9.5 <= size <= 10.5 and y < 360).replace("\n", "").strip()
    return title, body

from langchain_core.documents import Document

def split_text(text, size, overlap):
    """size 글자마다 자르고, 다음 조각은 앞 조각의 끝 overlap 글자부터 시작한다 → (시작, 끝) 목록"""
    assert 0 <= overlap < size, "겹침은 0 이상, 조각 크기보다 작아야 합니다"
    spans, start = [], 0
    while True:
        end = min(start + size, len(text))
        spans.append((start, end))
        if end >= len(text):
            return spans
        start += size - overlap

def make_chunks(title, body, size, overlap):
    return [Document(page_content=body[s:e], metadata={
                "chunk_id": f"pwc-p{PDF_PAGE}-c{i:02d}", "source": SOURCE, "title": title, "file": PDF_FILE.name,
                "book_page": BOOK_PAGE, "pdf_page": PDF_PAGE, "url": VIEW_URL, "start": s, "end": e,
                "chunk_size": size, "overlap": overlap})
            for i, (s, e) in enumerate(split_text(body, size, overlap), start=1)]

import hashlib
from langchain_chroma import Chroma

def open_index(chunks):
    """설정(조각 크기·겹침)·본문·임베딩 모델이 같으면 같은 컬렉션 이름 → 다시 실행해도 중복 저장하지 않는다."""
    m = chunks[0].metadata
    key = EMBEDDING_MODEL_NAME + "".join(c.page_content for c in chunks)
    name = f"pwc-p{PDF_PAGE}-{m['chunk_size']}-{m['overlap']}-{hashlib.sha256(key.encode('utf-8')).hexdigest()[:8]}"
    store = Chroma(collection_name=name, embedding_function=embeddings, persist_directory="chroma_db",
                   collection_metadata={"hnsw:space": "cosine"})            # 거리 = 코사인 거리(작을수록 가까움)
    ids = [c.metadata["chunk_id"] for c in chunks]
    have = set(store.get(ids=ids)["ids"])
    new = [c for c in chunks if c.metadata["chunk_id"] not in have]
    if new:
        store.add_documents(new, ids=[c.metadata["chunk_id"] for c in new])
    print(f"컬렉션 {name} · 새로 추가 {len(new)}개 · 저장된 조각 {len(store.get()['ids'])}개")
    return store

def search(store, question, k=3, **kw):
    found = store.similarity_search_with_score(question, k=k, **kw)
    return [{"rank": i, "distance": round(float(d), 4), "text": doc.page_content, **doc.metadata}
            for i, (doc, d) in enumerate(found, start=1)]

def show_hits(hits):
    for h in hits:
        print(f"{h['rank']}위 [{h['chunk_id']}] 거리 {h['distance']} · {h['source']} 책자 {h['book_page']}쪽 = PDF {h['pdf_page']}번째 페이지"
              f" · 본문 {h['start']}~{h['end']}자")
        print("    " + h["text"])

reader = PdfReader(str(ensure_pdf()))
TITLE, BODY = read_body(reader.pages[PAGE_INDEX])
CHUNKS = make_chunks(TITLE, BODY, 200, 60)
store = open_index(CHUNKS)
print(f"문서: {SOURCE} · 책자 {BOOK_PAGE}쪽(PDF {PDF_PAGE}번째 페이지) · ‘{TITLE}’ · 본문 {len(BODY)}자 · 조각 {len(CHUNKS)}개")

## 2. State — 질문 한 건을 처리하며 무엇을 기억할까?

State는 노드들이 함께 읽고 쓰는 **한 번의 실행 기록**입니다. 처음에는 `question`만 있고, 노드를 지날 때마다 필드가 채워집니다.

| 시점 | `question` | `route` | `documents` | `answer` | `sources` |
|---|---|---|---|---|---|
| 시작 | 감광액이 덮이지 않은 부분을 없애는 단계는? | — | — | — | — |
| 질문 분류 뒤 | (그대로) | `document` | — | — | — |
| 문서 검색 뒤 | (그대로) | (그대로) | 검색된 조각 3개 | — | — |
| 근거 답변 뒤 | (그대로) | (그대로) | (그대로) | 답변 | 확인된 인용·쪽수 |

- **이번 실행의 State**: 질문 하나를 처리하는 동안만 존재합니다. 다음 질문은 새 State로 시작합니다.
- **대화 기록(메모리)**: 여러 질문에 걸쳐 남기는 정보입니다. 이 노트북에서는 다루지 않습니다(체크포인터 등 별도 장치가 필요).

In [ ]:
from typing import TypedDict

class RAGState(TypedDict, total=False):   # total=False: 처음에는 question만 있어도 된다
    question: str        # 사용자 질문
    route: str           # "general" 또는 "document"
    documents: list      # 검색된 조각(원문·chunk_id·쪽수)
    answer: str          # 최종 답변
    sources: list        # 확인된 인용과 쪽수

start = RAGState(question="감광액이 덮이지 않은 부분을 없애는 단계는?")
print("시작 State:", start)

노드는 **바꾼 필드만** 돌려주면 됩니다. LangGraph가 그 값을 기존 State에 합칩니다. 모델 없이 한 노드짜리 그래프로 확인합니다.

In [ ]:
from langgraph.graph import END, START, StateGraph

def fake_classify(state: RAGState) -> dict:
    return {"route": "document"}          # question은 돌려주지 않는다

demo = StateGraph(RAGState)
demo.add_node("classify", fake_classify)
demo.add_edge(START, "classify")
demo.add_edge("classify", END)
print("노드가 돌려준 값: {'route': 'document'}")
print("실행 후 State:   ", demo.compile().invoke(start))   # question이 그대로 남아 있다

## 3. Node — 각 단계에서 어떤 일을 할까?

노드는 State를 받아 **갱신할 필드만 dict로 돌려주는** 평범한 Python 함수입니다. 각 함수 위의 한 줄은 `읽는 필드 → 하는 일 → 갱신하는 필드`입니다.

먼저 모델 응답에서 글자만 꺼내는 도우미와, 검색 호출 횟수를 세는 기록을 둡니다(일반 대화 경로에서 검색이 0회인지 확인용).

In [ ]:
import json

def text_of(content):
    """모델 응답 content가 문자열이면 그대로, 블록 목록(추론 블록 + 글자 블록)이면 글자 블록만 이어 붙인다."""
    if isinstance(content, str):
        return content
    return "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")

SEARCH_LOG = []   # retrieve_documents가 Chroma를 부를 때마다 질문을 남긴다

import json

def read_json(text):
    text = str(text).strip()
    a, b = text.find("{"), text.rfind("}")
    try:
        return json.loads(text[a:b + 1]) if a >= 0 and b > a else {}
    except json.JSONDecodeError:
        return {}

def squash(text):
    return "".join(str(text).split())

def check_citations(data, hits):
    """모델이 댄 조각 ID가 검색 결과에 있고, 인용구가 그 조각 원문에 글자 그대로 있는지 코드로 확인한다."""
    by_id = {h["chunk_id"]: h for h in hits}
    rows = []
    for s in data.get("sentences", []):
        h = by_id.get(s.get("chunk_id"))
        ok = bool(h) and bool(s.get("quote")) and squash(s["quote"]) in squash(h["text"])
        rows.append({**s, "verified": ok, "source": (f"{h['source']} 책자 {h['book_page']}쪽 = PDF {h['pdf_page']}번째 페이지"
                                                       if h else "검색 결과에 없는 조각"), "url": h["url"] if h else ""})
    return rows

### 질문 분류 — 모델은 두 값 중 하나만 고른다

`json_llm`(JSON 객체 하나만 답하는 모델)에 `{"route": ...}` 형식을 요구하고, **코드가 다시 확인**합니다. 두 값이 아니거나 JSON을 읽을 수 없으면 일반 대화로 넘기지 않고 오류로 멈춥니다(잘못된 분류가 조용히 숨지 않게).

In [ ]:
ROUTES = ("general", "document")
CLASSIFY_SYSTEM = """당신은 반도체 공정 챗봇의 질문 분류기입니다. JSON 객체 하나만 출력합니다: {"route": "general" 또는 "document"}
- general: 인사·감사·잡담·챗봇 사용법처럼 공정 자료가 필요 없는 말
- document: 반도체 공정·장비·불량·실험 등 자료를 찾아 근거로 답해야 하는 질문(자료에 답이 없을 것 같아도 document)"""

# question → 채팅 모델로 general/document 선택 → route
def classify_question(state: RAGState) -> dict:
    reply = json_llm.invoke([("system", CLASSIFY_SYSTEM), ("human", state["question"])])
    raw = text_of(reply.content)
    route = read_json(raw).get("route")
    if route not in ROUTES:
        raise ValueError(f"분류 결과를 읽지 못했습니다(일반 대화로 처리하지 않음): {raw[:200]!r}")
    return {"route": route}

In [ ]:
DIRECT_SYSTEM = "당신은 반도체 공정 챗봇입니다. 인사·잡담에는 한국어로 1~2문장으로 짧게 답하고, 공정 질문은 편하게 물어보라고 안내합니다."

# question → 문서 없이 바로 답변 → answer, sources(빈 목록)
def answer_directly(state: RAGState) -> dict:
    reply = llm.invoke([("system", DIRECT_SYSTEM), ("human", state["question"])])
    return {"answer": text_of(reply.content).strip(), "sources": []}

In [ ]:
# question → Chroma에서 관련 조각 3개 검색 → documents(원문·chunk_id·쪽수 포함)
def retrieve_documents(state: RAGState) -> dict:
    SEARCH_LOG.append(state["question"])
    return {"documents": search(store, state["question"], k=3)}

In [ ]:
RAG_SYSTEM = """당신은 반도체 공정 자료를 안내하는 도우미입니다. 한국어로 3~4문장 이내로 쉽게 답합니다.
[검색된 원문]에 있는 내용만 근거로 답하고, 원문에 없는 사실은 쓰지 않습니다.
JSON 객체 하나만 출력합니다.
{"answerable": true 또는 false,
 "sentences": [{"text": "답변 문장", "chunk_id": "근거 조각 ID", "quote": "그 조각 원문에서 글자 그대로 옮긴 짧은 구절"}],
 "missing": ["원문으로 확인할 수 없는 질문의 부분"]}
- 답변 문장마다 그 문장을 뒷받침하는 조각 ID 하나와 인용구를 붙입니다. 인용구는 원문을 바꾸지 말고 그대로 옮깁니다.
- 질문 일부만 원문으로 답할 수 있으면 답할 수 있는 문장만 쓰고, 나머지는 missing에 적습니다.
- 원문으로 답할 수 없으면 answerable을 false, sentences를 []로 둡니다.
- 질문이 묻는 핵심(특정 회사·라인·날짜의 사실 등)이 원문에 없으면, 관련 배경 설명이 있더라도 answerable을 false로 둡니다."""

def format_context(hits):
    return "\n\n".join(f"[조각ID: {h['chunk_id']}] {h['source']} 책자 {h['book_page']}쪽(PDF {h['pdf_page']}쪽)\n{h['text']}" for h in hits)

def make_message(question, hits):
    return f"[질문]\n{question}\n\n[검색된 원문]\n{format_context(hits)}"

NOT_FOUND = "제공된 문서에서 확인할 수 없습니다."

# question, documents → 검색 원문만 근거로 답변, 인용구를 원문과 대조 → answer, sources
def answer_with_evidence(state: RAGState) -> dict:
    hits = state["documents"]
    reply = json_llm.invoke([("system", RAG_SYSTEM), ("human", make_message(state["question"], hits))])
    data = read_json(text_of(reply.content))
    rows = [r for r in check_citations(data, hits) if r["verified"]]   # 원문에 없는 인용은 버린다
    if not data.get("answerable") or not rows:
        missing = " · ".join(data.get("missing") or [state["question"]])
        return {"answer": f"{NOT_FOUND} (근거 부족: {missing})", "sources": []}
    answer = " ".join(f"{r['text']} [{i}]" for i, r in enumerate(rows, start=1))
    sources = [{"n": i, "chunk_id": r["chunk_id"], "quote": r["quote"], "page": r["source"], "url": r["url"]}
               for i, r in enumerate(rows, start=1)]
    return {"answer": answer, "sources": sources}

## 4. Edge — 다음에 실행할 노드는 어떻게 정할까?

```
START → 질문 분류(classify)
          ├─ route = "general"  → 직접 답변(direct)  → END
          └─ route = "document" → 문서 검색(retrieve) → 근거 답변(grounded) → END
```

- **고정 엣지** `add_edge`: 항상 다음 노드로 갑니다. `문서 검색 → 근거 답변`, `START → 질문 분류`, 각 답변 → `END`.
- **조건부 엣지** `add_conditional_edges`: 함수가 돌려준 값으로 갈림길을 고릅니다. `질문 분류 → 직접 답변 또는 문서 검색`.

분류 모델은 `classify_question`에서 **한 번만** 호출됩니다. 조건부 엣지의 `lambda`는 이미 State에 저장된 `route`를 읽기만 합니다.

In [ ]:
builder = StateGraph(RAGState)
builder.add_node("classify", classify_question)
builder.add_node("direct", answer_directly)
builder.add_node("retrieve", retrieve_documents)
builder.add_node("grounded", answer_with_evidence)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    lambda state: state["route"],
    {"general": "direct", "document": "retrieve"},
)
builder.add_edge("direct", END)
builder.add_edge("retrieve", "grounded")
builder.add_edge("grounded", END)

graph = builder.compile()

# 외부 렌더링 없이 컴파일된 그래프의 실제 연결을 글자로 출력한다
for e in graph.get_graph().edges:
    kind = f"조건부(route = {e.data})" if e.conditional else "고정"
    print(f"{e.source:>9} → {e.target:<9} {kind}")

### 워크플로우를 그림으로 보고 PNG로 받기

`graph.get_graph()`가 컴파일된 그래프 구조를 Mermaid 그림 문법으로 바꾸고, `draw_mermaid_png()`가 그 그림을 PNG로 만듭니다. PNG는 노트북 옆 `langgraph_workflow.png`로 저장되고, 아래 링크로 내려받을 수 있습니다.

- PNG 변환은 공개 서비스 mermaid.ink에 **그림 문법(노드 이름과 연결)만** 보냅니다. 질문·문서·키는 보내지 않습니다.
- 외부 접속이 막힌 환경이면 그림 대신 Mermaid 문법을 출력합니다. 위의 글자 연결도는 그대로 쓸 수 있습니다.

In [ ]:
import base64
from pathlib import Path
from IPython.display import HTML, Image, display

flow = graph.get_graph()
try:
    png = flow.draw_mermaid_png()                       # 컴파일된 그래프 → PNG 바이트
    path = Path("langgraph_workflow.png")
    path.write_bytes(png)                               # 노트북 옆에 저장
    display(Image(png))
    link = base64.b64encode(png).decode()
    display(HTML(f'<a download="{path.name}" href="data:image/png;base64,{link}">PNG 내려받기 ({path.name}, {len(png):,}바이트)</a>'))
    print("저장:", path.resolve())
except Exception as e:                                  # 외부 접속이 막혔을 때
    print(f"PNG를 만들지 못했습니다({type(e).__name__}). 아래 Mermaid 문법을 https://mermaid.live 등에 붙여 그림으로 볼 수 있습니다.\n")
    print(flow.draw_mermaid())

### `graph.invoke()` — 질문 하나의 최종 State

`invoke`는 그래프를 끝까지 실행하고 **마지막 State 하나**를 돌려줍니다. 어떤 노드를 거쳤는지는 5절의 `stream`으로 봅니다.

In [ ]:
final = graph.invoke({"question": "감광액이 덮이지 않은 부분을 없애는 단계는?"})
for key in ("question", "route", "answer", "sources"):
    print(f"{key:9}:", final.get(key))
print("documents: 조각", [d["chunk_id"] for d in final.get("documents", [])])

## 5. 실행 비교 — 세 질문이 실제로 거친 경로

`graph.stream(..., stream_mode="updates")`는 노드가 끝날 때마다 `{노드 이름: 그 노드가 갱신한 값}`을 하나씩 내보냅니다. 질문마다 **한 번만 실행**하고, 그 실행에서 나온 노드 순서·답변·출처를 함께 기록합니다(같은 질문을 다시 부르면 모델 응답이 달라질 수 있어 기록을 섞지 않음).

In [ ]:
RESULTS = []

def short(value):
    if isinstance(value, list) and value and isinstance(value[0], dict) and "chunk_id" in value[0]:
        return [v["chunk_id"] for v in value]
    return value

def run(question):
    SEARCH_LOG.clear()
    nodes, state = [], {"question": question}
    print("질문:", question)
    for update in graph.stream({"question": question}, stream_mode="updates"):
        for node, changed in update.items():
            nodes.append(node)
            state.update(changed)
            print(f"  [{node}] 갱신:", {k: short(v) for k, v in changed.items()})
    RESULTS.append({"question": question, "nodes": nodes, "searches": len(SEARCH_LOG),
                    "answer": state.get("answer", ""), "sources": state.get("sources", [])})
    print("  실행 노드:", " → ".join(nodes), f"· Chroma 검색 {len(SEARCH_LOG)}회")
    print("  최종 답변:", state.get("answer"))
    for s in state.get("sources", []):
        print(f"  [{s['n']}] “{s['quote']}” · {s['chunk_id']} · {s['page']}")

In [ ]:
run("안녕하세요?")

In [ ]:
run("감광액이 덮이지 않은 부분을 없애는 단계는?")

In [ ]:
run("우리 회사 A라인의 어제 식각 불량 원인은 뭐야?")

### 한 표로 비교 — 위 세 실행의 기록에서 만든 표

In [ ]:
from IPython.display import Markdown, display

def cell(text):
    return str(text).replace("|", "\\|").replace("\n", " ")

lines = ["| 질문 | 실제 실행 노드 순서 | 최종 답변 | 인용·출처 |", "|---|---|---|---|"]
for r in RESULTS:
    cites = "<br>".join(f"[{s['n']}] “{s['quote']}” · {s['chunk_id']} · {s['page']}" for s in r["sources"]) or "없음"
    lines.append(f"| {cell(r['question'])} | {' → '.join(r['nodes'])} (검색 {r['searches']}회) | {cell(r['answer'])} | {cell(cites)} |")
display(Markdown("\n".join(lines)))

### 확인할 것

- **안녕하세요?** 실행 노드에 `retrieve`가 없고 Chroma 검색이 0회인가?
- **감광액이 덮이지 않은 부분을 없애는 단계는?** 인용구가 실제로 검색된 조각에 있고, 쪽수가 책자 6쪽 = PDF 8번째 페이지인가? (맨 위 발행처 원문 링크로 대조)
- **우리 회사 A라인의 어제 식각 불량 원인은 뭐야?** `retrieve`를 거쳐 검색은 했는데 근거 부족으로 끝났는가? 붙은 출처가 없는가?
- 맨 위에 적은 내 예상과 실제 노드 순서가 같은가? 다르면 어느 노드의 갱신값(`route`) 때문인가?

에이전트(앞 실습)는 모델이 도구 호출 여부를 정했고, 이 그래프는 **개발자가 정한 두 경로** 중 하나를 분류 결과로 고릅니다. 경로가 고정돼 있어 검색 생략·근거 확인 같은 단계를 빠뜨리지 않습니다.